# 9장 — AI 채점관의 성적표: LLM 심판 ⚖️🤖

서술형 답변 수만 개는 사람이 못 읽습니다 → **LLM 심판**(AI 채점관)의 시대.
그런데 심판도 모델이라 **편향**이 있어요. 측정 도구부터 감정해야 합니다.

신의 시점 설계: 200판 대결, 진짜 A 승률 **54.5%**(품질 차 g = U(−1,1)+0.1).
심판 점수 = g + **위치**(먼저 보인 답 +0.3) + **길이**(긴 답 +0.2·d) + **자기 선호**(형제 모델 B에 +0.15).

In [ ]:
# 🤖 AI 튜터 — 공부하다 막히면 여기서 물어보세요 (로컬 Ollama, 없어도 학습 지장 없음)
from tutor import chat
chat('ch9')

## 1. 무대 만들기: 웹 랩과 같은 난수열

In [ ]:
M32 = 0xFFFFFFFF

def imul(a, b):
    return (a * b) & M32

def mulberry32(seed):
    a = seed & M32
    def rand():
        nonlocal a
        a = (a + 0x6D2B79F5) & M32
        t = a
        t = imul(t ^ (t >> 15), t | 1) & M32
        t = (t ^ ((t + imul(t ^ (t >> 7), t | 61)) & M32)) & M32
        return (t ^ (t >> 14)) / 4294967296
    return rand

N = 200
rand = mulberry32(900)
G, D = [], []
for _ in range(N):
    G.append(rand() * 2 - 1 + 0.1)
    D.append(rand() * 2 - 1)

TRUE_WINS = sum(1 for g in G if g > 0)
print(f'진짜 A 승수: {TRUE_WINS}/200 = {TRUE_WINS/2:.1f}%')
assert TRUE_WINS == 109, '웹 랩과 동일 — 진짜 승률 54.5%'

B_POS, B_LEN, B_SELF = 0.3, 0.2, 0.15

def verdicts(pos, ln, self_, a_first):
    out = []
    for i in range(N):
        s = G[i] + (B_POS * (1 if a_first else -1) if pos else 0) \
            + (B_LEN * D[i] if ln else 0) + (-B_SELF if self_ else 0)
        out.append(s > 0)
    return out

def agree(v):
    a = 0.0
    for i in range(N):
        truth = G[i] > 0
        a += 0.5 if v[i] is None else (1 if v[i] == truth else 0)
    return a / N * 100

## 2. 편향별 지문: 단일 채점(A를 항상 먼저)

In [ ]:
scenarios = [
    ('편향 없음  ', False, False, False),
    ('위치만    ', True, False, False),
    ('길이만    ', False, True, False),
    ('자기선호만 ', False, False, True),
    ('전부 켬   ', True, True, True),
]
print('시나리오   | A 승률 | 사람 일치율')
print('-' * 40)
results = {}
for name, p, l, s in scenarios:
    v = verdicts(p, l, s, True)
    rate = sum(v) / 2
    results[name.strip()] = (rate, agree(v))
    print(f'{name} | {rate:5.1f}% | {agree(v):5.1f}%')

assert results['위치만'] == (67.0, 87.5), '위치 하나로 +12.5%p'
assert results['자기선호만'][0] == 47.5, '우위(54.5%)가 열세로 둔갑'
assert results['길이만'] == (54.5, 95.0), '집계는 그대로, 개별 판정 10판이 뒤집힘'
assert results['전부 켬'][0] == 60.5
print()
print('✔ 편향마다 지문이 다르다: 부풀림/둔갑/조용한 오심')

## 3. 교차 채점: 순서를 뒤집어 두 번 (위치 소거)

두 순서의 판정이 일치하면 채택, 갈리면 **판정 불가**(0.5 처리)로 둡니다. 불일치율 자체가
위치 편향의 측정기입니다. 단, **자기 선호는 교차로 못 지웁니다.**

In [ ]:
def crossed(pos, ln, self_):
    v1 = verdicts(pos, ln, self_, True)
    v2 = verdicts(pos, ln, self_, False)
    out, wins, und = [], 0, 0
    for i in range(N):
        if v1[i] == v2[i]:
            out.append(v1[i])
            wins += v1[i]
        else:
            out.append(None)
            und += 1
    return (wins + und * 0.5) / N * 100, und, out

for name, p, l, s in [('전부 켬  ', True, True, True), ('위치만   ', True, False, False)]:
    rate, und, v = crossed(p, l, s)
    print(f'{name}: 교차 A 승률 {rate:.1f}% (판정 불가 {und}판 = {und/2:.0f}%, 사람 일치율 {agree(v):.1f}%)')

r_all, und_all, _ = crossed(True, True, True)
r_pos, und_pos, _ = crossed(True, False, False)
assert (r_all, und_all) == (45.0, 62), '자기선호가 남아 여전히 왜곡'
assert (r_pos, und_pos) == (53.0, 56), '위치만이면 교차로 진짜(54.5)에 근접'
print()
print('✔ 교차는 위치만 지운다. 자기 선호는 심판·선수 분리(구조)로 대응')

# 심판 간 일치의 함정 — J2(위치 반대 성향 −0.3, 길이 0.3, 자기선호 없음)
def verdicts_j2(a_first):
    return [G[i] + -0.3 * (1 if a_first else -1) + 0.3 * D[i] > 0 for i in range(N)]

v1 = verdicts(True, True, True, True)
v2 = verdicts_j2(True)
inter = sum(1 for i in range(N) if v1[i] == v2[i]) / 2
print(f'\n심판 간 일치(J1↔J2): {inter:.1f}%  vs  사람과 J1 {agree(v1):.1f}% / J2 {agree(v2):.1f}%')
assert inter == 77.0
print('✔ 심판끼리의 일치가 사람 대비보다 낮다. 일치 ≠ 정답 (편향 공유 가능)')

## 4. 그림: 편향별 왜곡 요약

In [ ]:
import matplotlib.pyplot as plt

try:
    plt.rcParams['font.family'] = 'Malgun Gothic'
    plt.rcParams['axes.unicode_minus'] = False
except Exception:
    pass

names = list(results.keys()) + ['교차(전부)', '교차(위치만)']
rates = [results[k][0] for k in results] + [r_all, r_pos]
colors = ['#10b981', '#fb7185', '#f59e0b', '#a78bfa', '#e11d48', '#6366f1', '#4f46e5']

plt.figure(figsize=(9, 3.8))
plt.bar(names, rates, color=colors)
plt.axhline(TRUE_WINS / 2, color='#0f172a', ls='--', lw=1.5, label=f'진짜 승률 {TRUE_WINS/2:.1f}%')
plt.ylabel('관측 A 승률 (%)')
plt.title('AI 심판의 성적표: 편향과 대책에 따라')
plt.legend()
plt.xticks(fontsize=8)
plt.tight_layout()
plt.show()
print('점선(진짜)에 가장 가까운 것: 편향 없음, 그리고 교차(위치만). 대책이 작동한다는 증거')

## 정리: 오늘 가져갈 세 문장

1. **심판도 모델이다**: 위치(+12.5%p)·길이(조용한 오심)·자기 선호(우위 둔갑)의 지문.
2. 대책은 짝으로: 위치→**교차 채점**(7장 짝지은 비교의 철학), 자기 선호→**심판·선수 분리**,
   길이→루브릭. 감정의 최종 기준은 **사람 채점 표본과의 일치율**(3부 10장 상관의 실전판).
3. 심판끼리의 일치는 정답의 보증이 아니고(편향 공유), 심판 승률도 **n판 위의 측정**이라
   6장 오차범위가 그대로 적용된다(200판 ≈ ±7%p).

다음 **10장(🎯 캡스톤)**: 가상의 신모델 발표를 통째로 팩트체크하며 졸업합니다.